# K-Nearest Neighbors Lesson (5/28/26)
---------------------------------------------------------------------------------------------------------------------------------------------------------

**K-Nearest Neighbors Classifier**

K-Nearest Neighbors Classifier

  - KNN is a classification algorithm — its core idea is that data points with similar attributes tend to fall into
  similar categories.
  - Data points are plotted by their attributes — e.g., each point has an x value and a y value, letting it be placed on
  a graph.
  - Color/label represents the class the algorithm is trying to predict (e.g., green or red).
  - White (uncolored) points have no class yet — classifying these unknown points is the algorithm's purpose.
  - k = the number of nearest neighbors the algorithm looks at to make a classification.
  - The value of k can change the result:
    - k = 3: smaller circle → 2 green, 1 red → classified as green
    - k = 5: larger circle → 3 red, 2 green → classified as red
  - Core process: given a dataset of points with known classes, take a new point with an unknown class, find its nearest
  neighbors, and classify it based on them.


**Introduction**


  - Feature — a piece of information associated with a data point.
  - Example dataset: movies. Potential numeric features of a movie:
    - Length of the movie (in minutes)
    - Budget of the movie (in dollars)
  - Numeric features let you place movies in a multi-dimensional space (like the 2D graph from before).
  - Boolean features — features that are either True or False. Examples:
    - Black and white → True for B&W movies, False otherwise
    - Directed by Stanley Kubrick → True only for his films, False for almost all others
  - Classification goal: Label each movie as good or bad.
    - "Good" = IMDb rating of 7.0 or greater → class 1
    - "Bad" = below 7.0 → class 0
  - Example data point format: [length, budget, directed_by_kubrick]

In [1]:
#Ex.) 

#Movie arrays 
mean_girls = [97, 17000000, False]
the_shining = [146, 19000000, True]
gone_with_the_wind = [238, 3977000, False]

**Distance Between Points - 2D**


  - Why we need a distance formula: humans can eyeball nearest neighbors on a graph, but a computer needs a precise
  definition of "close" vs. "far apart."
  - Solution: use the Distance Formula to measure how far apart two points are.
  - Example uses 2 dimensions:
    - Length of the movie
    - Movie's release date
  - Example data points:
    - Star Wars → 125 minutes, released 1977
    - Raiders of the Lost Ark → 115 minutes, released 1981
  - 2D distance formula:

  $$d = \sqrt{(x_1 - x_2)^2 + (y_1 - y_2)^2}$$


In [2]:
#Ex.) 

#Distance Formula Function
def distance(movie1, movie2):
  dist = ((movie1[0] - movie2[0])**2 + (movie1[1] - movie2[1])**2)**0.5
  return dist

#List of movies with arguments (Runtime, Release Date)
star_wars = [125, 1977]
raiders = [115, 1981]
mean_girls = [97, 2004]

#Calling and executing a function 
print("Distance of SW-Raiders: ",  distance(star_wars, raiders))
print("Distance of SW-MG: ",  distance(star_wars, mean_girls))

Distance of SW-Raiders:  10.770329614269007
Distance of SW-MG:  38.897300677553446


**Distance Between Points - 3D & ND**

 - Limitation of 2D: using only length and release date is restrictive — there's much more useful movie data available.
  - Add a third dimension: e.g., the movie's budget, requiring distance to be measured in 3D space.
  - 3D distance formula:

  $$d = \sqrt{(x_1 - x_2)^2 + (y_1 - y_2)^2 + (z_1 - z_2)^2}$$

  - Beyond 3D: points become impossible to visualize past 3 dimensions, but distance can still be calculated.
  - Generalized distance formula between points $A$ and $B$ in $N$-dimensional space:

  $$d = \sqrt{(A_1 - B_1)^2 + (A_2 - B_2)^2 + \cdots + (A_n - B_n)^2}$$

  - $A_1 - B_1$ = difference between the first feature of each point
  - $A_n - B_n$ = difference between the last feature of each point
  - Takeaway: this lets us find the K-Nearest Neighbors of a point in any number of dimensions, using as many movie
  features as we want.
  - These distances will later be used to find the nearest neighbors of an unlabeled point.

In [3]:
#Ex.) 

star_wars = [125, 1977, 11000000]
raiders = [115, 1981, 18000000]
mean_girls = [97, 2004, 17000000]

def distance1(movie1, movie2):
  squared_difference = 0
  for i in range(len(movie1)):
    squared_difference += (movie1[i] - movie2[i])**2
  Final_Dist = (squared_difference)**0.5
  return Final_Dist 

print("Distance of SW-Raiders: ",  distance1(star_wars, raiders))
print("Distance of SW-MG: ",  distance1(star_wars, mean_girls))

Distance of SW-Raiders:  7000000.000008286
Distance of SW-MG:  6000000.000126083


**Data with Different Scales: Normalization**


- The three steps of the KNN algorithm:
    a. Normalize the data
    b. Find the k nearest neighbors
    c. Classify the new point based on those neighbors
  - The problem — dimensions have very different scales:
    - Release dates differ by at most ~125 years
    - Budgets can differ by millions of dollars
  - Why this breaks KNN: the distance formula treats all dimensions equally regardless of scale.
    - A 1-year difference is treated as equal to a $1 budget difference — absurd.
    - The large-scale feature (budget) dominates and drowns out all other dimensions, making them essentially
  meaningless.
  - The solution — normalize the data so every value falls between 0 and 1.
  - This lesson uses min-max normalization.
  - Min-max normalization formula:

  $$x_{norm} = \frac{x - \min}{\max - \min}$$

In [4]:
#Ex.) 

release_dates = [1897.0, 1998.0, 2000.0, 1948.0, 1962.0, 1950.0, 1975.0, 1960.0, 2017.0, 1937.0, 1968.0, 1996.0, 1944.0, 1891.0, 1995.0, 1948.0, 2011.0, 1965.0, 1891.0, 1978.0]

def min_max_normalize(lst):
    minimum = min(lst)
    maximum = max(lst)
    normalized = []
    for value in lst:
      normalized.append((value - minimum) / (maximum - minimum))
    return normalized

print(min_max_normalize(release_dates))

[0.047619047619047616, 0.8492063492063492, 0.8650793650793651, 0.4523809523809524, 0.5634920634920635, 0.46825396825396826, 0.6666666666666666, 0.5476190476190477, 1.0, 0.36507936507936506, 0.6111111111111112, 0.8333333333333334, 0.42063492063492064, 0.0, 0.8253968253968254, 0.4523809523809524, 0.9523809523809523, 0.5873015873015873, 0.0, 0.6904761904761905]


  - Because 1891 is the minimum and 2017 is the maximum in the list. Min-max
    normalization maps the smallest value to 0 and the largest to 1. Since 1897 is only 6 years above the minimum (out of
    a 126-year total range), it lands very close to 0.

**Finding The Nearest Neighbors**

  # The 3 steps of the KNN algorithm
  - Normalize the data (so all features are on the same scale)
  - Find the *k* nearest neighbors of the unknown point
  - Classify the new point based on the labels of those neighbors

  # Finding the k nearest neighbors
  - Compare the new, unclassified point to **every** other point in the dataset
  - Use the distance formula repeatedly — once per existing point
  - Store each result as `[distance, label]` pairs
  - **Sort** the list by distance (smallest → largest)
  - Keep the first *k* entries — these are the nearest neighbors

  # Example output (sorted distances)
  - `[0.30, 'Superman II']` → closest neighbor
  - `[0.31, 'Finding Nemo']`
  - `...`
  - `[0.38, 'Blazing Saddles']` → farther away
  - Smaller distance = more similar to the unknown point

  # Key terms
  - **k** = how many neighbors to look at (here, 5 — chosen somewhat arbitrarily for now)
  - Choosing *k* well is covered in a later exercise
  - The neighbors' **labels** are what we'll use next to classify the unknown point


In [5]:
#Ex.) 

"""


from movies import movie_dataset, movie_labels

print(movie_dataset['Bruce Almighty'])
print(movie_labels['Bruce Almighty'])

def distance(movie1, movie2):
  squared_difference = 0
  for i in range(len(movie1)):
    squared_difference += (movie1[i] - movie2[i]) ** 2
  final_distance = squared_difference ** 0.5
  return final_distance

def classify(unknown, dataset, k):
  distances = []
  #Loop through every title in dataset 
  for title in dataset:
    movie = dataset[title]
    distance_to_point = distance(movie, unknown)
    #adding to list and ordering
    distances.append([distance_to_point, title])
    distances.sort()
  #Getting the k-nearest neighbors
  neighbors = distances[0:k]
  return neighbors
    

#Testing the classify function 
print(classify([.4, .2, .9], movie_dataset, 5))
    
"""



"\n\n\nfrom movies import movie_dataset, movie_labels\n\nprint(movie_dataset['Bruce Almighty'])\nprint(movie_labels['Bruce Almighty'])\n\ndef distance(movie1, movie2):\n  squared_difference = 0\n  for i in range(len(movie1)):\n    squared_difference += (movie1[i] - movie2[i]) ** 2\n  final_distance = squared_difference ** 0.5\n  return final_distance\n\ndef classify(unknown, dataset, k):\n  distances = []\n  #Loop through every title in dataset \n  for title in dataset:\n    movie = dataset[title]\n    distance_to_point = distance(movie, unknown)\n    #adding to list and ordering\n    distances.append([distance_to_point, title])\n    distances.sort()\n  #Getting the k-nearest neighbors\n  neighbors = distances[0:k]\n  return neighbors\n    \n\n#Testing the classify function \nprint(classify([.4, .2, .9], movie_dataset, 5))\n    \n"

**Count Neighbors**


  ## Where we are in the algorithm
  -  Normalize the data
  -  Find the k nearest neighbors
  -  Classify the new point based on those neighbors ← this step

  ## The neighbors list looks like this
  - `[0.083, 'Lady Vengeance']`
  - `[0.236, 'Steamboy']`
  - `...`
  - `[0.331, 'Godzilla 2000']`
  - Each entry = `[distance, title]`, already sorted closest → farthest

  ## The goal: vote
  - Count how many neighbors are good vs. bad
  - More good → classify unknown movie as good
  - More bad → classify unknown movie as bad

  ## How to find each neighbor's class
  - Use the `movie_labels` dataset
  - `movie_labels[title]` → returns the label
    - $1$ = good movie (e.g. `movie_labels['Akira']` → $1$)
    - $0$ = bad movie
  - Loop through the neighbors, look up each title's label, and tally the count

  ## Handling a tie
  - Possible when $k$ is even (e.g. $k = 8$, with 4 good + 4 bad)
  - Different strategies exist; one common tie-breaker:
  - Choose the class of the closest point (first item in the sorted list)




In [7]:
#Ex.) 

"""

from movies import movie_dataset, movie_labels

def distance(movie1, movie2):
  squared_difference = 0
  for i in range(len(movie1)):
    squared_difference += (movie1[i] - movie2[i]) ** 2
  final_distance = squared_difference ** 0.5
  return final_distance

def classify(unknown, dataset, labels, k):
  num_good = 0
  num_bad =  0 
  distances = []
  #Looping through all points in the dataset
  for title in dataset:
    movie = dataset[title]
    distance_to_point = distance(movie, unknown)
    #Adding the distance and point associated with that distance
    distances.append([distance_to_point, title])
  distances.sort()
  #Taking only the k closest points
  neighbors = distances[0:k]
  #Looping through titles in neighbors 
  for movie in neighbors:
    title = movie[1]
    if labels[title] == 0:
      num_bad += 1
    else: 
      num_good += 1
  if num_good > num_bad: 
    return 1
  else:
    return 0

#Testing the classify function 
print(classify([.4, .2, .9], movie_dataset, movie_labels, 5))

"""

'\n\nfrom movies import movie_dataset, movie_labels\n\ndef distance(movie1, movie2):\n  squared_difference = 0\n  for i in range(len(movie1)):\n    squared_difference += (movie1[i] - movie2[i]) ** 2\n  final_distance = squared_difference ** 0.5\n  return final_distance\n\ndef classify(unknown, dataset, labels, k):\n  num_good = 0\n  num_bad =  0 \n  distances = []\n  #Looping through all points in the dataset\n  for title in dataset:\n    movie = dataset[title]\n    distance_to_point = distance(movie, unknown)\n    #Adding the distance and point associated with that distance\n    distances.append([distance_to_point, title])\n  distances.sort()\n  #Taking only the k closest points\n  neighbors = distances[0:k]\n  #Looping through titles in neighbors \n  for movie in neighbors:\n    title = movie[1]\n    if labels[title] == 0:\n      num_bad += 1\n    else: \n      num_good += 1\n  if num_good > num_bad: \n    return 1\n  else:\n    return 0\n\n#Testing the classify function \nprint(clas

**Classify Your Favorite Movie**

In [8]:
# Ex.) 


"""


from movies import movie_dataset, movie_labels, normalize_point

def distance(movie1, movie2):
  squared_difference = 0
  for i in range(len(movie1)):
    squared_difference += (movie1[i] - movie2[i]) ** 2
  final_distance = squared_difference ** 0.5
  return final_distance

def classify(unknown, dataset, labels, k):
  distances = []
  #Looping through all points in the dataset
  for title in dataset:
    movie = dataset[title]
    distance_to_point = distance(movie, unknown)
    #Adding the distance and point associated with that distance
    distances.append([distance_to_point, title])
  distances.sort()
  #Taking only the k closest points
  neighbors = distances[0:k]
  num_good = 0
  num_bad = 0
  for neighbor in neighbors:
    title = neighbor[1]
    if labels[title] == 0:
      num_bad += 1
    elif labels[title] == 1:
      num_good += 1
  if num_good > num_bad:
    return 1
  else:
    return 0

print("Call Me By Your Name" in movie_dataset)
my_movie = [3500000, 132, 2017]
normalized_my_movie = normalize_point(my_movie)
print("Norm: ", normalized_my_movie)
print("K-Mean: ", classify(normalized_my_movie, movie_dataset, movie_labels, 5))


"""

'\n\n\nfrom movies import movie_dataset, movie_labels, normalize_point\n\ndef distance(movie1, movie2):\n  squared_difference = 0\n  for i in range(len(movie1)):\n    squared_difference += (movie1[i] - movie2[i]) ** 2\n  final_distance = squared_difference ** 0.5\n  return final_distance\n\ndef classify(unknown, dataset, labels, k):\n  distances = []\n  #Looping through all points in the dataset\n  for title in dataset:\n    movie = dataset[title]\n    distance_to_point = distance(movie, unknown)\n    #Adding the distance and point associated with that distance\n    distances.append([distance_to_point, title])\n  distances.sort()\n  #Taking only the k closest points\n  neighbors = distances[0:k]\n  num_good = 0\n  num_bad = 0\n  for neighbor in neighbors:\n    title = neighbor[1]\n    if labels[title] == 0:\n      num_bad += 1\n    elif labels[title] == 1:\n      num_good += 1\n  if num_good > num_bad:\n    return 1\n  else:\n    return 0\n\nprint("Call Me By Your Name" in movie_datase

**Training and Validation Sets**

 ## Why we need this step
  - You've built a working KNN classifier — it can predict if a movie's IMDb rating is above or below $7.0$
  - But a working algorithm isn't necessarily a correct one
  - We need to measure how effective it actually is — predictions could be totally wrong

  ## Splitting the data
  - Like most machine learning algorithms, split the data into two sets:
    - Training set → the "known" movies the classifier compares against
    - Validation set → held-out movies used to test the classifier
  - The validation set acts as a stand-in for never-before-seen data

  ## How validation works
  - Take each movie from the validation set, one at a time
  - Treat it as the unknown point: compare it to all movies in the training set
  - Find its K nearest neighbors and make a prediction
  - Peek at the real answer in the validation labels to check if the prediction was correct

  ## Computing validation accuracy
  - Repeat for every movie in the validation set
  - Count how many predictions were right and how many were wrong
  - $$\text{validation accuracy} = \frac{\text{number correct}}{\text{total in validation set}}$$

  ## Why it matters
  - Validation accuracy changes depending on which $K$ you choose
  - Next exercise: use validation accuracy to pick the best possible $K$ for the classifier

  - Key idea to hold onto: the validation set's labels are never used to make the prediction — only to grade it afterward. That's what makes the accuracy score honest.

In [9]:
# Ex.)

"""


from movies import training_set, training_labels, validation_set, validation_labels

def distance(movie1, movie2):
  squared_difference = 0
  for i in range(len(movie1)):
    squared_difference += (movie1[i] - movie2[i]) ** 2
  final_distance = squared_difference ** 0.5
  return final_distance

def classify(unknown, dataset, labels, k):
  distances = []
  #Looping through all points in the dataset
  for title in dataset:
    movie = dataset[title]
    distance_to_point = distance(movie, unknown)
    #Adding the distance and point associated with that distance
    distances.append([distance_to_point, title])
  distances.sort()
  #Taking only the k closest points
  neighbors = distances[0:k]
  num_good = 0
  num_bad = 0
  for neighbor in neighbors:
    title = neighbor[1]
    if labels[title] == 0:
      num_bad += 1
    elif labels[title] == 1:
      num_good += 1
  if num_good > num_bad:
    return 1
  else:
    return 0


print(validation_set["Bee Movie"])
guess = classify(validation_set["Bee Movie"], training_set, training_labels, 5)
print(guess)
if guess == validation_labels["Bee Movie"]:
  print("Correct!")





"""

'\n\n\nfrom movies import training_set, training_labels, validation_set, validation_labels\n\ndef distance(movie1, movie2):\n  squared_difference = 0\n  for i in range(len(movie1)):\n    squared_difference += (movie1[i] - movie2[i]) ** 2\n  final_distance = squared_difference ** 0.5\n  return final_distance\n\ndef classify(unknown, dataset, labels, k):\n  distances = []\n  #Looping through all points in the dataset\n  for title in dataset:\n    movie = dataset[title]\n    distance_to_point = distance(movie, unknown)\n    #Adding the distance and point associated with that distance\n    distances.append([distance_to_point, title])\n  distances.sort()\n  #Taking only the k closest points\n  neighbors = distances[0:k]\n  num_good = 0\n  num_bad = 0\n  for neighbor in neighbors:\n    title = neighbor[1]\n    if labels[title] == 0:\n      num_bad += 1\n    elif labels[title] == 1:\n      num_good += 1\n  if num_good > num_bad:\n    return 1\n  else:\n    return 0\n\n\nprint(validation_set["

**Choosing k**

 ## The tradeoff
  - Validation accuracy changes as $k$ changes
  - The goal: find a $k$ that's neither too small nor too large
  - Too small → overfitting; too large → underfitting

  ## Overfitting (k too small, e.g. $k = 1$)
  - Relying too heavily on the training data
  - Assumes real-world data will behave exactly like the training set
  - In KNN: happens when you don't consider enough neighbors
  - A single outlier can drastically determine an unknown point's label
  - Example: a lone dark-blue outlier makes every nearby point get classified dark blue, when it should be green
  - The classifier has latched onto small quirks/noise in the training data
  - Result: low validation accuracy

  ## Underfitting (k too large)
  - The classifier doesn't pay enough attention to the small quirks in the training set
  - Extreme example: 100 training points and $k = 100$
    - Every unknown point gets classified the same way
    - The distances between points stop mattering at all
  - The classifier loses any real understanding of the training data
  - Result: low validation accuracy (for the opposite reason)

  ## Key takeaway
  - Both extremes hurt accuracy:
    - $k$ too small → too sensitive to noise (overfit)
    - $k$ too large → too insensitive to structure (underfit)
  - The best $k$ usually sits somewhere in the middle — found by testing validation accuracy across many values of $k$

  - Mental model: small $k$ = "listens to one loud neighbor" (gets misled by outliers); large $k$ = "polls the whole town" (drowns out the local signal). You're tuning $k$ to balance the two.

In [10]:
# Ex.) 


"""

from movies import training_set, training_labels, validation_set, validation_labels

def distance(movie1, movie2):
  squared_difference = 0
  for i in range(len(movie1)):
    squared_difference += (movie1[i] - movie2[i]) ** 2
  final_distance = squared_difference ** 0.5
  return final_distance

def classify(unknown, dataset, labels, k):
  distances = []
  #Looping through all points in the dataset
  for title in dataset:
    movie = dataset[title]
    distance_to_point = distance(movie, unknown)
    #Adding the distance and point associated with that distance
    distances.append([distance_to_point, title])
  distances.sort()
  #Taking only the k closest points
  neighbors = distances[0:k]
  num_good = 0
  num_bad = 0
  for neighbor in neighbors:
    title = neighbor[1]
    if labels[title] == 0:
      num_bad += 1
    elif labels[title] == 1:
      num_good += 1
  if num_good > num_bad:
    return 1
  else:
    return 0


def find_validation_accuracy(training_set, training_labels, validation_set, validation_labels, k):
  num_correct = 0.0 
  for title in validation_set:
    guess = classify(validation_set[title], training_set, training_labels, k)
    if guess == validation_labels[title]:
      num_correct += 1
  return (num_correct/len(validation_set))

print("Val Acc for K = 3: ",  find_validation_accuracy(training_set, training_labels, validation_set, validation_labels, 3))


"""

'\n\nfrom movies import training_set, training_labels, validation_set, validation_labels\n\ndef distance(movie1, movie2):\n  squared_difference = 0\n  for i in range(len(movie1)):\n    squared_difference += (movie1[i] - movie2[i]) ** 2\n  final_distance = squared_difference ** 0.5\n  return final_distance\n\ndef classify(unknown, dataset, labels, k):\n  distances = []\n  #Looping through all points in the dataset\n  for title in dataset:\n    movie = dataset[title]\n    distance_to_point = distance(movie, unknown)\n    #Adding the distance and point associated with that distance\n    distances.append([distance_to_point, title])\n  distances.sort()\n  #Taking only the k closest points\n  neighbors = distances[0:k]\n  num_good = 0\n  num_bad = 0\n  for neighbor in neighbors:\n    title = neighbor[1]\n    if labels[title] == 0:\n      num_bad += 1\n    elif labels[title] == 1:\n      num_good += 1\n  if num_good > num_bad:\n    return 1\n  else:\n    return 0\n\n\ndef find_validation_accu

**Graph of K**

* Best K in Val_acc vs K is k = 75 

**Using sklearn**


  ## Why use sklearn
  - You've built KNN from scratch — but you don't need to rewrite it every time
  - `sklearn` is a Python library built specifically for machine learning
  - It has many features; here we use only its K-Nearest Neighbor classifier

  ## Step 1 — Create the classifier object
  - Use `KNeighborsClassifier`, which takes one parameter: $k$
  - Passed in as `n_neighbors`

In [11]:
classifier = KNeighborsClassifier(n_neighbors = 3)

NameError: name 'KNeighborsClassifier' is not defined

 ## Step 2 — Train the classifier with .fit()
  - `.fit()` takes two parameters:
    - a list of points (the training data)
    - the labels for those points
  - The two lists must line up by index (point $i$ ↔ label $i$)

In [13]:
training_points = [
    [0.5, 0.2, 0.1],
    [0.9, 0.7, 0.3],
    [0.4, 0.5, 0.7]
  ]


training_labels = [0, 1, 1]
classifier.fit(training_points, training_labels)


NameError: name 'classifier' is not defined

  ## Step 3 — Classify new points with .predict()
  - `.predict()` takes a list of points you want to classify
  - Returns a list of guesses — one label per input point

In [14]:
  unknown_points = [
    [0.2, 0.1, 0.7],
    [0.4, 0.7, 0.6],
    [0.5, 0.8, 0.1]
  ]

guesses = classifier.predict(unknown_points)

NameError: name 'classifier' is not defined

  ## The pattern to remember (most sklearn models share it)
  - Create → `KNeighborsClassifier(n_neighbors = k)`
  - Train → `.fit(points, labels)`
  - Predict → `.predict(new_points)`
  - Don't forget to import it first: `from sklearn.neighbors import KNeighborsClassifier`


In [15]:
# Ex.) 

"""

rom movies import movie_dataset, labels
from sklearn.neighbors import KNeighborsClassifier

classifier = KNeighborsClassifier(n_neighbors = 5)
classifier.fit(movie_dataset, labels)

unknown_points = [[.45, .2, .5],  [.25, .8, .9], [.1, .1, .9]]
guesses = classifier.predict(unknown_points)
print(guesses)

"""

'\n\nrom movies import movie_dataset, labels\nfrom sklearn.neighbors import KNeighborsClassifier\n\nclassifier = KNeighborsClassifier(n_neighbors = 5)\nclassifier.fit(movie_dataset, labels)\n\nunknown_points = [[.45, .2, .5],  [.25, .8, .9], [.1, .1, .9]]\nguesses = classifier.predict(unknown_points)\nprint(guesses)\n\n'

**Review**

 ## What you accomplished
  - Built a classifier from scratch
  - Reimplemented the same classifier using Python's `sklearn` library
  - Learned techniques specific to KNN, plus general machine learning ideas

  ## Major takeaways

  ### Data as points in space
  - Data with $n$ features can be conceptualized as points lying in $n$-dimensional space

  ### Comparing points
  - Data points are compared using the distance formula
  - Similar points have a smaller distance between them

  ### Classifying
  - A point with an unknown class is classified by finding its $k$ nearest neighbors

  ### Measuring effectiveness
  - Split data with known classes into a training set and a validation set
  - Use the validation set to calculate validation error / accuracy

  ### Tuning parameters
  - Classifiers have parameters that can be tuned to improve performance
  - For KNN, that parameter is $k$

  ### Overfitting vs. underfitting
  - A classifier can be trained improperly:
    - Low $k$ → overfitting (too sensitive to noise/outliers)
    - Large $k$ → underfitting (ignores meaningful local structure)

  ### Libraries
  - `sklearn` can be used for many classification and machine learning algorithms
  - The create → fit → predict pattern transfers to other models

  ## Interactive demo (in the lesson)
  - Moving the mouse over the canvas classifies that location as green or blue
  - The nearest neighbors are highlighted in yellow
  - The slider changes $k$ — watch how the classification boundaries shift as $k$ grows

  - That's the full KNN module wrapped up. Nice work getting through it. Want a single condensed "cheat sheet" cell summarizing the whole lesson (distance → neighbors → vote → validate → tune $k$ → sklearn)
  for quick reference later?